# 04z — Build master gustos

LEFT JOIN secuencial de los 6 parquets de features sobre `sample_user_ids_gustos.parquet`.

In [1]:
import pandas as pd
import numpy as np
import time
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '03_master' / '04z_build_master'
INFORMES.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {INFORMES}')


Output dir: /Users/jezquerro/Documents/tfg/informes/fase4_gustos/03_master/04z_build_master


In [2]:
# 1. Cargar sample base
t0 = time.time()
sample = pd.read_parquet(DATA_QC / 'sample_user_ids_gustos.parquet')
N = len(sample)
print(f'Sample base: {N:,}')
master = sample[['user_id']].copy()

# 2. LEFT JOINs
FEATURE_PARQUETS = [
    'features_reutilizadas_churn.parquet',
    'features_proporciones.parquet',
    'features_diversidad.parquet',
    'features_temporales.parquet',
    'features_intensidad.parquet',
    'features_monetizacion.parquet',
]

overlap_log = []
join_log = []
for parquet_name in FEATURE_PARQUETS:
    df = pd.read_parquet(DATA_QC / parquet_name)
    overlap = (set(master.columns) & set(df.columns)) - {'user_id'}
    if overlap:
        overlap_log.append({'parquet': parquet_name, 'overlap_cols': list(overlap)})
        print(f'  Overlap en {parquet_name}: {overlap}')
        # Renombrar para evitar colision (preferir versión del primer parquet)
        df = df.drop(columns=list(overlap))
    n_before = master.shape[1]
    master = master.merge(df, on='user_id', how='left')
    join_log.append({'parquet': parquet_name, 'cols_added': master.shape[1] - n_before, 'master_shape': master.shape})
    print(f'  {parquet_name}: +{master.shape[1]-n_before} cols → master {master.shape}')

# Sanity checks
assert len(master) == N, f'N cambio: {len(master)} vs {N}'
assert master['user_id'].is_unique, 'user_id duplicados'
assert master['user_id'].notna().all(), 'user_id NaN'

print(f'\nMaster shape: {master.shape}')
print(f'  Tiempo: {time.time()-t0:.1f}s')


Sample base: 114,412
  features_reutilizadas_churn.parquet: +70 cols → master (114412, 71)
  features_proporciones.parquet: +23 cols → master (114412, 94)
  features_diversidad.parquet: +8 cols → master (114412, 102)
  features_temporales.parquet: +10 cols → master (114412, 112)


  features_intensidad.parquet: +6 cols → master (114412, 118)
  features_monetizacion.parquet: +5 cols → master (114412, 123)

Master shape: (114412, 123)
  Tiempo: 0.2s


In [3]:
# Guardar master sin cleanup
master.to_parquet(DATA_QC / 'master_table_gustos.parquet', index=False)
print(f'Guardado: master_table_gustos.parquet ({master.shape})')

# Save schema JSON
import json
schema = {
    'shape': list(master.shape),
    'columns': master.columns.tolist(),
    'dtypes': {c: str(master[c].dtype) for c in master.columns},
}
with open(DATA_QC / 'master_table_gustos_schema.json', 'w') as f:
    json.dump(schema, f, indent=2)


Guardado: master_table_gustos.parquet ((114412, 123))


In [4]:
# Reporte ejecutivo
nan_rates = (master.drop(columns=['user_id']).isna().mean()*100).round(2)
nan_sorted = nan_rates.sort_values(ascending=False)

dtype_counts = master.dtypes.astype(str).value_counts()

lines = [
    '# 04z — Build master gustos',
    '',
    f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Shape**: {master.shape}',
    f'**Tiempo**: {time.time()-t0:.1f}s',
    '',
    '## Distribución de tipos',
    '',
    '| dtype | n_columnas |',
    '|---|---:|',
]
for dt, n in dtype_counts.items():
    lines.append(f'| {dt} | {n} |')

lines += ['', '## Overlap de columnas entre parquets', '']
if overlap_log:
    for entry in overlap_log:
        lines.append(f'- `{entry["parquet"]}`: {entry["overlap_cols"]}')
    lines.append('')
    lines.append('Resolución: las columnas duplicadas se descartan del parquet posterior (se conserva la versión del primer parquet en aparecer).')
else:
    lines.append('- Ninguno. Todos los parquets aportaron columnas únicas.')

lines += ['', '## Log de LEFT JOINs', '', '| Parquet | Cols añadidas | Shape resultante |', '|---|---:|---|']
for entry in join_log:
    lines.append(f'| {entry["parquet"]} | {entry["cols_added"]} | {entry["master_shape"]} |')

lines += ['', f'## Top 30 features con más NaN', '', '| Feature | %NaN |', '|---|---:|']
for f, v in nan_sorted.head(30).items():
    lines.append(f'| {f} | {v} |')

lines += ['', f'## Top 30 features con menos NaN', '', '| Feature | %NaN |', '|---|---:|']
for f, v in nan_sorted.tail(30).iloc[::-1].items():
    lines.append(f'| {f} | {v} |')

lines += ['', '## Estadísticas globales de NaN', '',
          f'- Features sin NaN: {(nan_rates==0).sum()}',
          f'- Features con NaN<5%: {(nan_rates<5).sum()}',
          f'- Features con NaN>50%: {(nan_rates>50).sum()}',
          f'- Features con NaN>80%: {(nan_rates>80).sum()}',
          f'- Features con NaN>95%: {(nan_rates>95).sum()}',
         ]

(INFORMES / 'execution_report.md').write_text('\n'.join(lines))
print('execution_report.md guardado')
print(f'\nMaster: {master.shape}')
print(f'Features sin NaN: {(nan_rates==0).sum()}')
print(f'Features con NaN<5%: {(nan_rates<5).sum()}')
print(f'Features con NaN>80%: {(nan_rates>80).sum()}')


execution_report.md guardado

Master: (114412, 123)
Features sin NaN: 83
Features con NaN<5%: 98
Features con NaN>80%: 24
